In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import requests

## Data Exploration

Before building our first pipeline and retrieving the selected dataset, some preliminary work is required.

### Objectives
- Request data from the API  
- Explore the data structure  
- Identify necessary transformations  

### Notes
Reviewing the API documentation is a key part of this phase, as it often answers questions that arise during exploration.

In [2]:
url = "https://opensky-network.org/api/states/all"
response = requests.get(url)
data = response.json()

In [3]:
aviation_raw = pd.DataFrame(data)
aviation_raw.head()

,time,states
0,1777094061,"[39de4e, TVF68FN , France, 1777094061, 1777094..."
1,1777094061,"[a4754b, N387A , United States, 1777093901, ..."
2,1777094061,"[e8027e, LPE2375 , Chile, 1777094060, 17770940..."
3,1777094061,"[39de4d, TVF83MR , France, 1777094060, 1777094..."
4,1777094061,"[39de4c, TVF45BX , France, 1777094061, 1777094..."


In [5]:
aviation_raw.shape

(5532, 2)

## Data Transformation & Structure Overview

After retrieving the data from the API in JSON format, it is converted into a Pandas DataFrame.

The first five rows and the shape of the dataset are displayed to get a quick overview of the structure.

According to the OpenSky API documentation, the data consists of:

- **Timestamp (Unix time):** Represents the time of the data snapshot; all records share this value.
- **State vector:** Contains real-time information about each aircraft.

In [6]:
aviation_raw.iloc[0]

time                                             1777094061
states    [39de4e, TVF68FN , France, 1777094061, 1777094...
Name: 0, dtype: object

In [10]:
aviation_raw["time"].unique()

array([1777094061], dtype=int64)

In [11]:
aviation_raw["states"].iloc[0]

['39de4e',
 'TVF68FN ',
 'France',
 1777094061,
 1777094061,
 2.8029,
 45.4906,
 11894.82,
 False,
 225.54,
 177.52,
 0,
 None,
 12115.8,
 '1000',
 False,
 0]

In [18]:
count_el_state_vektor = len(aviation_raw["states"].iloc[0])
print(f"Number of elements per state vector: {len(aviation_raw["states"].iloc[0])}")


Number of elements per statevektor: 17


After verifying that all timestamps in the dataset are identical, the state vector was explored by selecting the first row, displaying its content, and counting the number of elements it contains.

The result does not fully match the schema described in the OpenSky API documentation. Further review shows that unauthenticated or restricted requests return only 17 fields instead of the full 18.

The 18th field is optional and is only included under specific conditions. For the current analysis, this additional feature is not required, so the dataset is used as provided.

In [20]:
# defining categories as metadata is not transmitted on requests
categories = ['icao240', 'callsign', 'origin_country', 'time_position', 'last_contact', 'longitude', 'latitude',
              'baro_altitude', 'on_ground', 'velocity', 'true_track', 'vertical_rate', 'sensors', 'geo_altitude',
              'squawk', 'spi', 'position_source'
             ]

In [39]:
# building dataframe from state vectors and setting columns 
states_data = pd.DataFrame(data["states"], columns=categories)

In [40]:
states_data

,icao240,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,39de4e,TVF68FN,France,1.777094e+09,1777094061,2.8029,45.4906,11894.82,False,225.54,177.52,0.00,None,12115.80,1000,False,0
1,a4754b,N387A,United States,1.777094e+09,1777093901,-97.8248,35.4822,449.58,False,51.96,0.00,-2.93,None,396.24,None,False,0
2,e8027e,LPE2375,Chile,1.777094e+09,1777094060,-59.0619,-34.6340,1988.82,False,146.35,132.72,-7.15,None,2095.50,5661,False,0
3,39de4d,TVF83MR,France,1.777094e+09,1777094060,1.1977,46.7669,8823.96,False,201.37,9.56,-2.93,None,9067.80,1000,False,0
4,39de4c,TVF45BX,France,1.777094e+09,1777094061,9.5428,44.0482,11582.40,False,225.04,332.50,0.33,None,11826.24,1000,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5527,a7f804,SKW5271,United States,1.777094e+09,1777094061,-122.3943,37.6271,548.64,False,72.01,296.75,10.73,None,518.16,None,False,0
5528,451cc1,ASL79H,Bulgaria,1.777094e+09,1777094061,19.3721,45.0392,6477.00,False,183.23,302.62,7.80,None,6659.88,None,False,0
5529,ad00c4,N937MG,United States,1.777094e+09,1777094061,-87.3683,39.9612,13106.40,False,264.33,86.76,0.00,None,13243.56,7267,False,0
5530,7c4811,JST688,Australia,1.777094e+09,1777094060,138.5646,-31.6530,10972.80,False,230.58,355.78,0.00,None,11414.76,1344,False,0


After expanding the state vectors into 17 columns, we add the corresponding timestamps to complete the structured DataFrame.

In [41]:
aviation_expanded = states_data.copy()
aviation_expanded.insert(0, "time", aviation_raw.time)

In [42]:
aviation_expanded

,time,icao240,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,sensors,geo_altitude,squawk,spi,position_source
0,1777094061,39de4e,TVF68FN,France,1.777094e+09,1777094061,2.8029,45.4906,11894.82,False,225.54,177.52,0.00,None,12115.80,1000,False,0
1,1777094061,a4754b,N387A,United States,1.777094e+09,1777093901,-97.8248,35.4822,449.58,False,51.96,0.00,-2.93,None,396.24,None,False,0
2,1777094061,e8027e,LPE2375,Chile,1.777094e+09,1777094060,-59.0619,-34.6340,1988.82,False,146.35,132.72,-7.15,None,2095.50,5661,False,0
3,1777094061,39de4d,TVF83MR,France,1.777094e+09,1777094060,1.1977,46.7669,8823.96,False,201.37,9.56,-2.93,None,9067.80,1000,False,0
4,1777094061,39de4c,TVF45BX,France,1.777094e+09,1777094061,9.5428,44.0482,11582.40,False,225.04,332.50,0.33,None,11826.24,1000,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5527,1777094061,a7f804,SKW5271,United States,1.777094e+09,1777094061,-122.3943,37.6271,548.64,False,72.01,296.75,10.73,None,518.16,None,False,0
5528,1777094061,451cc1,ASL79H,Bulgaria,1.777094e+09,1777094061,19.3721,45.0392,6477.00,False,183.23,302.62,7.80,None,6659.88,None,False,0
5529,1777094061,ad00c4,N937MG,United States,1.777094e+09,1777094061,-87.3683,39.9612,13106.40,False,264.33,86.76,0.00,None,13243.56,7267,False,0
5530,1777094061,7c4811,JST688,Australia,1.777094e+09,1777094060,138.5646,-31.6530,10972.80,False,230.58,355.78,0.00,None,11414.76,1344,False,0
